# Final Project: Data Analysis using Spark

Estimated time needed: **60** minutes

This final project is similar to the Practice Project you did. In this project, you will not be provided with hints or solutions. You will create a DataFrame by loading data from a CSV file and apply transformations and actions using Spark SQL. This needs to be achieved by performing the following tasks:

- Task 1: Generate DataFrame from CSV data.
- Task 2: Define a schema for the data.
- Task 3: Display schema of DataFrame.
- Task 4: Create a temporary view.
- Task 5: Execute an SQL query.
- Task 6: Calculate Average Salary by Department.
- Task 7: Filter and Display IT Department Employees.
- Task 8: Add 10% Bonus to Salaries.
- Task 9: Find Maximum Salary by Age.
- Task 10: Self-Join on Employee Data.
- Task 11: Calculate Average Employee Age.
- Task 12: Calculate Total Salary by Department.
- Task 13: Sort Data by Age and Salary.
- Task 14: Count Employees in Each Department.
- Task 15: Filter Employees with the letter o in the Name.


In [1]:
!pip install pyspark  findspark wget

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=6b38a7c50555d2f6c15b4f476bb1a32b3680c6c963d876948ad4f9f1f75919ec
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [2]:
import findspark
findspark.init()

In [3]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

In [4]:
sc = SparkContext.getOrCreate()
spark = SparkSession \
    .builder \
    .appName("Python Spark DataFrames basic example") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()

In [5]:
import wget
wget.download("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-BD0225EN-SkillsNetwork/data/employees.csv")

'employees.csv'

### Tasks


#### Task 1: Generate a Spark DataFrame from the CSV data

Read data from the provided CSV file, `employees.csv` and import it into a Spark DataFrame variable named `employees_df`.




In [6]:
# Read data from the "emp" CSV file and import it into a DataFrame variable named "employees_df"
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType

import pandas as pd
employees_data = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-BD0225EN-SkillsNetwork/data/employees.csv")
employees_data.head(2)
employees_df = spark.read.csv("employees.csv", header=True, inferSchema=True)

#### Task 2: Define a schema for the data

Construct a schema for the input data and then utilize the defined schema to read the CSV file to create a DataFrame named `employees_df`.


In [8]:
# Define a Schema for the input data and read the file using the user-defined Schema
schema = StructType([
    StructField("Emp_No", IntegerType(), True),
    StructField("Emp_Name", StringType(), True),
    StructField("Salary", IntegerType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Department", StringType(), True)
])

employees_df = spark.read.csv("employees.csv", header=True, schema=schema)

#### Task 3: Display schema of DataFrame

Display the schema of the `employees_df` DataFrame, showing all columns and their respective data types.


In [9]:
# Display all columns of the DataFrame, along with their respective data types
print("Schema of the Spark DataFrame:")
employees_df.printSchema()

Schema of the Spark DataFrame:
root
 |-- Emp_No: integer (nullable = true)
 |-- Emp_Name: string (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Department: string (nullable = true)



#### Task 4: Create a temporary view

Create a temporary view named `employees` for the `employees_df` DataFrame, enabling Spark SQL queries on the data.


In [10]:
# Create a temporary view named "employees" for the DataFrame
employees_df.createOrReplaceTempView("employees")

#### Task 5: Execute an SQL query

Compose and execute an SQL query to fetch the records from the `employees` view where the age of employees exceeds 30. Then, display the result of the SQL query, showcasing the filtered records.


In [11]:
# SQL query to fetch solely the records from the View where the age exceeds 30
spark.sql("""
SELECT * FROM employees WHERE Age > 30
""").show()

+------+-----------+------+---+----------+
|Emp_No|   Emp_Name|Salary|Age|Department|
+------+-----------+------+---+----------+
|   199|    Douglas|  2600| 34|     Sales|
|   200|   Jennifer|  4400| 36| Marketing|
|   201|    Michael| 13000| 32|        IT|
|   202|        Pat|  6000| 39|        HR|
|   203|      Susan|  6500| 36| Marketing|
|   205|    Shelley| 12008| 33|   Finance|
|   206|    William|  8300| 37|        IT|
|   100|     Steven| 24000| 39|        IT|
|   102|        Lex| 17000| 37| Marketing|
|   103|  Alexander|  9000| 39| Marketing|
|   104|      Bruce|  6000| 38|        IT|
|   105|      David|  4800| 39|        IT|
|   106|      Valli|  4800| 38|     Sales|
|   107|      Diana|  4200| 35|     Sales|
|   109|     Daniel|  9000| 35|        HR|
|   110|       John|  8200| 31| Marketing|
|   111|     Ismael|  7700| 32|        IT|
|   112|Jose Manuel|  7800| 34|        HR|
|   113|       Luis|  6900| 34|     Sales|
|   116|     Shelli|  2900| 37|   Finance|
+------+---

#### Task 6: Calculate Average Salary by Department

Compose an SQL query to retrieve the average salary of employees grouped by department. Display the result.


In [12]:
# SQL query to calculate the average salary of employees grouped by department
spark.sql("""
  SELECT Department,AVG(Salary) AS Avg_Salary
    FROM employees
    GROUP BY Department
""").show()

+----------+-----------------+
|Department|       Avg_Salary|
+----------+-----------------+
|     Sales|5492.923076923077|
|        HR|           5837.5|
|   Finance|           5730.8|
| Marketing|6633.333333333333|
|        IT|           7400.0|
+----------+-----------------+



#### Task 7: Filter and Display IT Department Employees

Apply a filter on the `employees_df` DataFrame to select records where the department is `'IT'`. Display the filtered DataFrame.


In [13]:
# Apply a filter to select records where the department is 'IT'
employees_df.filter(employees_df["Department"] == "IT").show()

+------+--------+------+---+----------+
|Emp_No|Emp_Name|Salary|Age|Department|
+------+--------+------+---+----------+
|   198|  Donald|  2600| 29|        IT|
|   201| Michael| 13000| 32|        IT|
|   206| William|  8300| 37|        IT|
|   100|  Steven| 24000| 39|        IT|
|   104|   Bruce|  6000| 38|        IT|
|   105|   David|  4800| 39|        IT|
|   111|  Ismael|  7700| 32|        IT|
|   129|   Laura|  3300| 38|        IT|
|   132|      TJ|  2100| 34|        IT|
|   136|   Hazel|  2200| 29|        IT|
+------+--------+------+---+----------+



#### Task 8: Add 10% Bonus to Salaries

Perform a transformation to add a new column named "SalaryAfterBonus" to the DataFrame. Calculate the new salary by adding a 10% bonus to each employee's salary.


In [14]:
from pyspark.sql.functions import col
employees_df = employees_df.withColumn(
    "SalaryAfterBonus",
    col("Salary") * 1.10
)

employees_df.show()

+------+---------+------+---+----------+------------------+
|Emp_No| Emp_Name|Salary|Age|Department|  SalaryAfterBonus|
+------+---------+------+---+----------+------------------+
|   198|   Donald|  2600| 29|        IT|2860.0000000000005|
|   199|  Douglas|  2600| 34|     Sales|2860.0000000000005|
|   200| Jennifer|  4400| 36| Marketing|            4840.0|
|   201|  Michael| 13000| 32|        IT|14300.000000000002|
|   202|      Pat|  6000| 39|        HR| 6600.000000000001|
|   203|    Susan|  6500| 36| Marketing| 7150.000000000001|
|   204|  Hermann| 10000| 29|   Finance|           11000.0|
|   205|  Shelley| 12008| 33|   Finance|13208.800000000001|
|   206|  William|  8300| 37|        IT|            9130.0|
|   100|   Steven| 24000| 39|        IT|26400.000000000004|
|   101|    Neena| 17000| 27|     Sales|           18700.0|
|   102|      Lex| 17000| 37| Marketing|           18700.0|
|   103|Alexander|  9000| 39| Marketing|            9900.0|
|   104|    Bruce|  6000| 38|        IT|

#### Task 9: Find Maximum Salary by Age

Group the data by age and calculate the maximum salary for each age group. Display the result.


In [15]:
from pyspark.sql.functions import max
spark.sql("""
  SELECT Age,MAX(Salary) AS Max_Salary
    FROM employees
    GROUP BY Age
""").show()

+---+----------+
|Age|Max_Salary|
+---+----------+
| 31|      8200|
| 34|      7800|
| 28|     12008|
| 27|     17000|
| 26|      3600|
| 37|     17000|
| 35|      9000|
| 39|     24000|
| 38|      6000|
| 29|     10000|
| 32|     13000|
| 33|     12008|
| 30|      8000|
| 36|      7900|
+---+----------+



#### Task 10: Self-Join on Employee Data

Join the "employees_df" DataFrame with itself based on the "Emp_No" column. Display the result.


In [16]:
# Join the DataFrame with itself based on the "Emp_No" column
employees_df.alias("e1") \
    .join(
        employees_df.alias("e2"),
        on="Emp_No",
        how="inner"
    ) \
    .show()

+------+---------+------+---+----------+------------------+---------+------+---+----------+------------------+
|Emp_No| Emp_Name|Salary|Age|Department|  SalaryAfterBonus| Emp_Name|Salary|Age|Department|  SalaryAfterBonus|
+------+---------+------+---+----------+------------------+---------+------+---+----------+------------------+
|   198|   Donald|  2600| 29|        IT|2860.0000000000005|   Donald|  2600| 29|        IT|2860.0000000000005|
|   199|  Douglas|  2600| 34|     Sales|2860.0000000000005|  Douglas|  2600| 34|     Sales|2860.0000000000005|
|   200| Jennifer|  4400| 36| Marketing|            4840.0| Jennifer|  4400| 36| Marketing|            4840.0|
|   201|  Michael| 13000| 32|        IT|14300.000000000002|  Michael| 13000| 32|        IT|14300.000000000002|
|   202|      Pat|  6000| 39|        HR| 6600.000000000001|      Pat|  6000| 39|        HR| 6600.000000000001|
|   203|    Susan|  6500| 36| Marketing| 7150.000000000001|    Susan|  6500| 36| Marketing| 7150.000000000001|
|

#### Task 11: Calculate Average Employee Age

Calculate the average age of employees using the built-in aggregation function. Display the result.


In [17]:
# Calculate the average age of employees
from pyspark.sql.functions import avg

employees_df.select(
    avg("Age").alias("Average_Age")
).show()


+-----------+
|Average_Age|
+-----------+
|      33.56|
+-----------+



#### Task 12: Calculate Total Salary by Department

Calculate the total salary for each department using the built-in aggregation function. Display the result.


In [18]:
# Calculate the total salary for each department.
from pyspark.sql.functions import sum
employees_df.groupBy("Department") \
    .agg(sum("Salary").alias("Total_Salary")) \
    .show()


+----------+------------+
|Department|Total_Salary|
+----------+------------+
|     Sales|       71408|
|        HR|       46700|
|   Finance|       57308|
| Marketing|       59700|
|        IT|       74000|
+----------+------------+



#### Task 13: Sort Data by Age and Salary

Apply a transformation to sort the DataFrame by age in ascending order and then by salary in descending order. Display the sorted DataFrame.


In [19]:
# Sort the DataFrame by age in ascending order and then by salary in descending order
employees_df.orderBy(
    "Age",
    employees_df["Salary"].desc()
).show()

+------+---------+------+---+----------+------------------+
|Emp_No| Emp_Name|Salary|Age|Department|  SalaryAfterBonus|
+------+---------+------+---+----------+------------------+
|   137|   Renske|  3600| 26| Marketing|3960.0000000000005|
|   101|    Neena| 17000| 27|     Sales|           18700.0|
|   114|      Den| 11000| 27|   Finance|12100.000000000002|
|   108|    Nancy| 12008| 28|     Sales|13208.800000000001|
|   130|    Mozhe|  2800| 28| Marketing|3080.0000000000005|
|   126|    Irene|  2700| 28|        HR|2970.0000000000005|
|   204|  Hermann| 10000| 29|   Finance|           11000.0|
|   115|Alexander|  3100| 29|   Finance|3410.0000000000005|
|   134|  Michael|  2900| 29|     Sales|3190.0000000000005|
|   198|   Donald|  2600| 29|        IT|2860.0000000000005|
|   140|   Joshua|  2500| 29|   Finance|            2750.0|
|   136|    Hazel|  2200| 29|        IT|            2420.0|
|   120|  Matthew|  8000| 30|        HR|            8800.0|
|   110|     John|  8200| 31| Marketing|

#### Task 14: Count Employees in Each Department

Calculate the number of employees in each department. Display the result.


In [20]:
from pyspark.sql.functions import count
employees_df.groupBy("Department") \
    .agg(count("*").alias("Employee_Count")) \
    .show()

+----------+--------------+
|Department|Employee_Count|
+----------+--------------+
|     Sales|            13|
|        HR|             8|
|   Finance|            10|
| Marketing|             9|
|        IT|            10|
+----------+--------------+



#### Task 15: Filter Employees with the letter o in the Name

Apply a filter to select records where the employee's name contains the letter `'o'`. Display the filtered DataFrame.


In [21]:
# Apply a filter to select records where the employee's name contains the letter 'o'
from pyspark.sql.functions import col

employees_df.filter(
    col("Emp_Name").contains("o")
).show()

+------+-----------+------+---+----------+------------------+
|Emp_No|   Emp_Name|Salary|Age|Department|  SalaryAfterBonus|
+------+-----------+------+---+----------+------------------+
|   198|     Donald|  2600| 29|        IT|2860.0000000000005|
|   199|    Douglas|  2600| 34|     Sales|2860.0000000000005|
|   110|       John|  8200| 31| Marketing|            9020.0|
|   112|Jose Manuel|  7800| 34|        HR|            8580.0|
|   130|      Mozhe|  2800| 28| Marketing|3080.0000000000005|
|   133|      Jason|  3300| 38|     Sales|3630.0000000000005|
|   139|       John|  2700| 36|     Sales|2970.0000000000005|
|   140|     Joshua|  2500| 29|   Finance|            2750.0|
+------+-----------+------+---+----------+------------------+



# Congratulations! You have completed the project.

Now you know how to create a DataFrame from a CSV data file and perform a variety of DataFrame transformations and actions using Spark SQL.


## Authors


Raghul Ramesh


Lavanya T S


<h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>
